# Fine-tuning BioMedVQA (BioMedCLIP + GPT-2) on ROCOv2 captioning

The BioMedVQA parallel to the BLIP-2 ROCO fine-tuning. It **resumes from the
VQA-RAD checkpoint** (`biomed_vqa_final.pt`) and further trains it on ROCOv2
captions, exactly like `biomed_captioner_v2.ipynb` but with two changes:

- the target is now a **caption** (prompt `"a photo of"`), not a VQA answer;
- we have a real **train / valid / test** split, so we **select the epoch on
  `valid`** and score **once** on `test`.

Only the translator + the last GPT-2 blocks + LM head train (BioMedCLIP frozen),
same as the VQA-RAD run. Evaluation uses the identical decoding + deberta
BERTScore as the zero-shot run, so the numbers are directly comparable.

> The full run is fast (~20 min), so it can run here directly, or headless via
> `train_roco_biomed.py` in `tmux`. The final **test** score is produced by
> `caption_roco_biomed.py` with `CKPT_PATH=<best epoch>`.

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")   # models are cached; skip flaky hub calls
import json, time, random
import torch
import torch.nn as nn
from PIL import Image

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device =", device)

device = cuda


In [2]:
# ── BioMedCLIP vision encoder + DETERMINISTIC eval transform ─────────────────
import open_clip
biomedclip_model, _preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    "hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224")
vision_encoder = biomedclip_model.visual
image_processor = preprocess_val   # deterministic Resize+CenterCrop (not the random-crop train transform)
print("BioMedCLIP vision encoder loaded")

/home/matei/miniconda3/envs/vlm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BioMedCLIP vision encoder loaded


## The model: same as VQA-RAD, now with a captioning forward

`BioMedVQA` prepends the image (one translated visual token) to the GPT-2 prompt.
`forward` computes the LM loss (with `-100` on the image slot); the dataset masks
the `"a photo of"` prompt so the loss lands only on the caption tokens + `</s>`.
Trainable: the translator + GPT-2 blocks 6--11 + `ln_f` + `lm_head`; the vision
encoder is frozen.

In [3]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

class BioMedVQA(nn.Module):
    def __init__(self, vision_encoder, text_model_name="gpt2"):
        super().__init__()
        self.vision_encoder = vision_encoder
        for p in self.vision_encoder.parameters():
            p.requires_grad = False
        self.llm = GPT2LMHeadModel.from_pretrained(text_model_name)
        TRAINABLE_BLOCKS = {6, 7, 8, 9, 10, 11}                 # train only the last 6 blocks + head
        for name, param in self.llm.named_parameters():
            block = next((int(x) for x in name.split(".") if x.isdigit()), None)
            param.requires_grad = (block in TRAINABLE_BLOCKS) or ("ln_f" in name) or ("lm_head" in name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(text_model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        self.translator = nn.Linear(512, self.llm.config.hidden_size)

    def forward(self, images, input_ids, attention_mask, labels=None):
        with torch.no_grad():
            image_features = self.vision_encoder(images)
        translated = self.translator(image_features).unsqueeze(1)              # [B,1,768]
        text_emb   = self.llm.transformer.wte(input_ids)                       # [B,L,768]
        inputs_embeds = torch.cat([translated, text_emb], dim=1)
        img_att = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
        attn    = torch.cat([img_att, attention_mask], dim=1)
        if labels is not None:
            img_lab = torch.full((labels.shape[0], 1), -100, device=labels.device)
            labels  = torch.cat([img_lab, labels], dim=1)
        return self.llm(inputs_embeds=inputs_embeds, attention_mask=attn, labels=labels)

    @torch.no_grad()
    def generate(self, images, input_ids, attention_mask, **gk):
        image_features = self.vision_encoder(images)
        translated = self.translator(image_features).unsqueeze(1)
        text_emb   = self.llm.transformer.wte(input_ids)
        inputs_embeds = torch.cat([translated, text_emb], dim=1)
        img_att = torch.ones((attention_mask.shape[0], 1), device=attention_mask.device)
        attn    = torch.cat([img_att, attention_mask], dim=1)
        return self.llm.generate(inputs_embeds=inputs_embeds, attention_mask=attn, **gk)

In [ ]:
# ── Build the model and RESUME from the VQA-RAD checkpoint ───────────────────
RESUME_FROM = "/home/matei/biomed_vqa_checkpoints/biomed_vqa_final.pt"
model = BioMedVQA(vision_encoder).to(device)
ckpt = torch.load(RESUME_FROM, map_location=device)
model.translator.load_state_dict(ckpt["translator"])
model.llm.load_state_dict(ckpt["llm"])
tokenizer = model.tokenizer
print(f"resumed from {RESUME_FROM}")
print(f"trainable {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M params")

In [ ]:
# ── ROCOv2 captioning data (train / valid subsets) ──────────────────────────
import pandas as pd
from torch.utils.data import Dataset, DataLoader

ROCO_DIR   = "/home/matei/rocov2"
CAP_PROMPT = "a photo of"
LLM_MAXLEN = 96      # 97% of captions fit untruncated
TRAIN_N    = 12000   # subset of the 59,958 train images
VAL_LOSS_N = 1000
VAL_GEN_N  = 500
BATCH      = 8

def load_roco(split, n=None, seed=0):
    caps = pd.read_csv(os.path.join(ROCO_DIR, f"{split}_captions.csv")).dropna(subset=["Caption"]).reset_index(drop=True)
    d = os.path.join(ROCO_DIR, split)
    recs = [{"id": r.ID, "path": os.path.join(d, f"{r.ID}.jpg"), "caption": str(r.Caption)}
            for r in caps.itertuples() if os.path.isfile(os.path.join(d, f"{r.ID}.jpg"))]
    if n is not None:
        random.Random(seed).shuffle(recs); recs = recs[:n]
    return recs

train_recs    = load_roco("train", TRAIN_N)
val_loss_recs = load_roco("valid", VAL_LOSS_N)
val_gen_recs  = load_roco("valid", VAL_GEN_N)
print(f"train {len(train_recs)} | val-loss {len(val_loss_recs)} | val-gen {len(val_gen_recs)}")

PROMPT_LEN = len(tokenizer(CAP_PROMPT).input_ids)   # GPT-2 has no BOS -> 3 tokens

class ROCOCaptionDataset(Dataset):
    def __init__(self, recs):
        self.recs = recs
    def __len__(self):
        return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        image = image_processor(Image.open(r["path"]).convert("RGB"))
        full  = f"{CAP_PROMPT} {r['caption']}{tokenizer.eos_token}"
        tok = tokenizer(full, padding="max_length", max_length=LLM_MAXLEN, truncation=True, return_tensors="pt")
        ids, att = tok.input_ids.squeeze(0), tok.attention_mask.squeeze(0)
        labels = ids.clone()
        labels[att == 0] = -100                        # ignore padding (real </s> keeps att==1 -> supervised)
        labels[:min(PROMPT_LEN, LLM_MAXLEN)] = -100    # ignore the "a photo of" prompt
        return {"images": image, "input_ids": ids, "attention_mask": att, "labels": labels}

train_loader = DataLoader(ROCOCaptionDataset(train_recs), batch_size=BATCH, shuffle=True, num_workers=2)
val_loader   = DataLoader(ROCOCaptionDataset(val_loss_recs), batch_size=BATCH, shuffle=False, num_workers=2)
print("prompt_len (masked):", PROMPT_LEN)

In [ ]:
# ── Validation generation + metrics (identical to caption_roco_biomed.py) ────
from tqdm.auto import tqdm
GEN_KWARGS = dict(max_new_tokens=40, min_new_tokens=8, num_beams=5, no_repeat_ngram_size=3,
                  length_penalty=1.0, eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)

def caption_records(recs, batch_size=16):
    tok = tokenizer(CAP_PROMPT, return_tensors="pt").to(device)
    preds = []
    for i in tqdm(range(0, len(recs), batch_size), desc="val gen", leave=False):
        chunk = recs[i:i+batch_size]
        pix = torch.stack([image_processor(Image.open(r["path"]).convert("RGB")) for r in chunk]).to(device)
        B = pix.shape[0]
        gen = model.generate(pix, tok.input_ids.expand(B, -1), tok.attention_mask.expand(B, -1), **GEN_KWARGS)
        preds.extend(s.strip() for s in tokenizer.batch_decode(gen, skip_special_tokens=True))
    return preds

@torch.no_grad()
def validation_loss(loader):
    model.eval()
    tot = n = 0
    for b in tqdm(loader, desc="val loss", leave=False):
        out = model(b["images"].to(device), b["input_ids"].to(device),
                    b["attention_mask"].to(device), b["labels"].to(device))
        k = b["images"].shape[0]; tot += float(out.loss) * k; n += k
    return tot / n

from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from nltk.translate.meteor_score import meteor_score
import transformers.modeling_utils as _mu
_mu.check_torch_load_is_safe = lambda *a, **k: None
from bert_score import BERTScorer
BERT_MODEL = "microsoft/deberta-xlarge-mnli"    # ROCOv2/ImageCLEF leaderboard model -> comparable
_bert_scorer = BERTScorer(model_type=BERT_MODEL, batch_size=8)
_bert_scorer._tokenizer.model_max_length = 512

def _norm(s):
    return " ".join(str(s).lower().split())

def compute_metrics(preds, refs):
    preds = [p if str(p).strip() else "." for p in preds]
    refs  = [r if str(r).strip() else "." for r in refs]
    gts = {i: [_norm(refs[i])]  for i in range(len(refs))}
    res = {i: [_norm(preds[i])] for i in range(len(preds))}
    bleu, _  = Bleu(4).compute_score(gts, res)
    rouge, _ = Rouge().compute_score(gts, res)
    cider, _ = Cider().compute_score(gts, res)
    meteor = sum(meteor_score([_norm(refs[i]).split()], _norm(preds[i]).split())
                 for i in range(len(preds))) / len(preds)
    _, _, F = _bert_scorer.score(preds, refs, batch_size=8, verbose=False)
    return {"BLEU-1": bleu[0], "BLEU-4": bleu[3], "METEOR": meteor, "ROUGE-L": rouge,
            "CIDEr": cider, "BERTScore-F1": F.mean().item()}
print("validation helpers ready")

In [ ]:
# ── Training loop: per-epoch checkpoint + validation selection ──────────────
from torch.optim import AdamW

EPOCHS, LR, WD = 3, 2e-5, 0.01
SAVE_DIR = "/home/matei/roco_biomed_checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

def save_ckpt(tag):
    p = os.path.join(SAVE_DIR, f"roco_biomed_{tag}.pt")
    torch.save({"translator": model.translator.state_dict(), "llm": model.llm.state_dict()}, p)
    return p

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WD)
history  = []
val_refs = [r["caption"] for r in val_gen_recs]

for epoch in range(EPOCHS):
    model.train()
    running, t0 = 0.0, time.time()
    for b in tqdm(train_loader, desc=f"epoch {epoch+1}/{EPOCHS}"):
        optimizer.zero_grad()
        out = model(b["images"].to(device), b["input_ids"].to(device),
                    b["attention_mask"].to(device), b["labels"].to(device))
        out.loss.backward(); optimizer.step()
        running += out.loss.item()
    train_loss = running / len(train_loader)

    path  = save_ckpt(f"epoch{epoch+1}")
    vloss = validation_loss(val_loader)
    model.eval()
    vpreds = caption_records(val_gen_recs)
    vm = compute_metrics(vpreds, val_refs)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "val_loss": vloss, "path": path, **vm})
    print(f"epoch {epoch+1} | {(time.time()-t0)/60:.1f} min | train {train_loss:.4f} | val {vloss:.4f} | "
          f"BERTScore {vm['BERTScore-F1']:.4f} | CIDEr {vm['CIDEr']:.4f} | ROUGE-L {vm['ROUGE-L']:.4f}")
    print(f"   REF : {val_refs[0][:90]}")
    print(f"   PRED: {vpreds[0][:90]}")
    print(f"   -> {path}")

In [ ]:
# ── Epoch selection on VALIDATION (test still untouched) ────────────────────
import pandas as pd
_cols = ["epoch", "train_loss", "val_loss", "BLEU-1", "BLEU-4", "METEOR", "ROUGE-L", "CIDEr", "BERTScore-F1"]
print(pd.DataFrame(history).reindex(columns=_cols).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

best = max(history, key=lambda r: r["BERTScore-F1"])   # ROCOv2's primary metric
print(f"\nBEST epoch by validation BERTScore: epoch {best['epoch']}  ->  {best['path']}")
print("\nFINAL STEP -- score it ONCE on the full test set (same script as zero-shot):")
print(f"  CKPT_PATH={best['path']} RUN_TAG=biomed_finetuned \\")
print(f"    /home/matei/miniconda3/envs/vlm/bin/python /home/matei/caption_roco_biomed.py")